# ============================================================
# CAPSTONE PROJECT
# NOTEBOOK 04: MONGODB DATA INGESTION PIPELINE
# ============================================================
# Purpose:
# This notebook creates a MongoDB pipeline for the FMA dataset.
# It loads metadata and features, prepares unified track records,
# and inserts them into MongoDB.
# ============================================================

In [1]:
# ============================================================
# 1. IMPORT LIBRARIES
# ============================================================

import pandas as pd
import numpy as np
import os
from pymongo import MongoClient


In [2]:
# ============================================================
# 2. SET PATHS
# ============================================================

metadata_path = "../data/raw/metadata"

tracks_path = os.path.join(metadata_path, "tracks.csv")
features_path = os.path.join(metadata_path, "features.csv")
genres_path = os.path.join(metadata_path, "genres.csv")


In [3]:

# ============================================================
# 3. LOAD DATA
# ============================================================

tracks = pd.read_csv(
    tracks_path,
    header=[0, 1],
    index_col=0
)

features = pd.read_csv(
    features_path,
    header=[0, 1, 2],
    index_col=0
)

genres = pd.read_csv(genres_path)

print("Tracks:", tracks.shape)
print("Features:", features.shape)
print("Genres:", genres.shape)


Tracks: (106574, 52)
Features: (106574, 518)
Genres: (163, 5)


In [4]:
# ============================================================
# 4. FILTER TO SMALL SUBSET FIRST
# ============================================================
# We start with small because it is safer for testing.
# Later, this same pipeline can be used for medium or large.

subset_name = "small"

tracks_subset = tracks[tracks[("set", "subset")] == subset_name]
features_subset = features.loc[tracks_subset.index]

print("Subset:", subset_name)
print("Tracks subset:", tracks_subset.shape)
print("Features subset:", features_subset.shape)

Subset: small
Tracks subset: (8000, 52)
Features subset: (8000, 518)


In [5]:
# ============================================================
# 5. KEEP ONLY TRACKS WITH GENRE LABELS
# ============================================================

valid = tracks_subset[("track", "genre_top")].notna()

tracks_subset = tracks_subset[valid]
features_subset = features_subset[valid]

print("After removing missing labels:")
print("Tracks:", tracks_subset.shape)
print("Features:", features_subset.shape)

After removing missing labels:
Tracks: (8000, 52)
Features: (8000, 518)


In [6]:
# ============================================================
# 6. CLEAN FEATURE COLUMNS
# ============================================================
# MongoDB does not work well with complex multi-index column names.
# We flatten feature columns into readable names.

features_subset = features_subset.select_dtypes(include=["number"])
features_subset = features_subset.replace([np.inf, -np.inf], np.nan)
features_subset = features_subset.fillna(features_subset.mean())

features_subset.columns = [
    "_".join([str(part) for part in col if str(part) != "nan"])
    for col in features_subset.columns
]

print("Cleaned features:", features_subset.shape)

Cleaned features: (8000, 518)


In [7]:
# ============================================================
# 7. FUNCTION TO CREATE AUDIO PATH
# ============================================================
# FMA audio files are stored using the track ID.
# Example:
# track_id = 2
# formatted = 000002
# folder = 000
# path = ../data/raw/audio/fma_small/000/000002.mp3

def create_audio_path(track_id, audio_base_path="../data/raw/audio/fma_small"):
    track_id_str = f"{int(track_id):06d}"
    folder = track_id_str[:3]
    return os.path.join(audio_base_path, folder, track_id_str + ".mp3")

In [8]:
# ============================================================
# 8. CREATE MONGODB RECORDS
# ============================================================
# Each record represents one track.
# This combines:
# - metadata from tracks.csv
# - split/subset information
# - genre label
# - audio path
# - selected numeric features

records = []

for track_id in tracks_subset.index:
    
    track_row = tracks_subset.loc[track_id]
    feature_row = features_subset.loc[track_id]
    
    record = {
        "track_id": int(track_id),
        "subset": str(track_row[("set", "subset")]),
        "split": str(track_row[("set", "split")]),
        
        "genre": {
            "genre_top": str(track_row[("track", "genre_top")]),
            "genres": str(track_row[("track", "genres")]),
            "genres_all": str(track_row[("track", "genres_all")])
        },
        
        "track_metadata": {
            "title": str(track_row[("track", "title")]),
            "duration": float(track_row[("track", "duration")]),
            "listens": int(track_row[("track", "listens")]),
            "interest": int(track_row[("track", "interest")]),
            "language_code": str(track_row[("track", "language_code")]),
            "license": str(track_row[("track", "license")])
        },
        
        "artist_metadata": {
            "artist_id": int(track_row[("artist", "id")]),
            "artist_name": str(track_row[("artist", "name")]),
            "artist_location": str(track_row[("artist", "location")])
        },
        
        "album_metadata": {
            "album_id": int(track_row[("album", "id")]),
            "album_title": str(track_row[("album", "title")]),
            "album_type": str(track_row[("album", "type")])
        },
        
        "audio_path": create_audio_path(track_id),
        
        "features": feature_row.to_dict()
    }
    
    records.append(record)

print("Total records prepared:", len(records))
print("Example record:")
records[0]

Total records prepared: 8000
Example record:


{'track_id': 2,
 'subset': 'small',
 'split': 'training',
 'genre': {'genre_top': 'Hip-Hop', 'genres': '[21]', 'genres_all': '[21]'},
 'track_metadata': {'title': 'Food',
  'duration': 168.0,
  'listens': 1293,
  'interest': 4656,
  'language_code': 'en',
  'license': 'Attribution-NonCommercial-ShareAlike 3.0 International'},
 'artist_metadata': {'artist_id': 1,
  'artist_name': 'AWOL',
  'artist_location': 'New Jersey'},
 'album_metadata': {'album_id': 1,
  'album_title': 'AWOL - A Way Of Life',
  'album_type': 'Album'},
 'audio_path': '../data/raw/audio/fma_small\\000\\000002.mp3',
 'features': {'chroma_cens_kurtosis_01': 7.1806526184,
  'chroma_cens_kurtosis_02': 5.2303090096,
  'chroma_cens_kurtosis_03': 0.24932080507,
  'chroma_cens_kurtosis_04': 1.3476201296,
  'chroma_cens_kurtosis_05': 1.4824777842,
  'chroma_cens_kurtosis_06': 0.53137123585,
  'chroma_cens_kurtosis_07': 1.4815930128,
  'chroma_cens_kurtosis_08': 2.691454649,
  'chroma_cens_kurtosis_09': 0.86686819792,
  'chrom

In [9]:
# ============================================================
# 9. CONNECT TO MONGODB
# ============================================================
# This assumes MongoDB is running locally.
# If using MongoDB Atlas, replace the connection string.

client = MongoClient("mongodb://localhost:27017/")

db = client["fma_capstone"]
collection = db["tracks_small"]

print("Connected to MongoDB successfully.")

Connected to MongoDB successfully.


In [10]:
# ============================================================
# 10. INSERT RECORDS INTO MONGODB
# ============================================================
# We delete old records first so the notebook can be rerun safely.

collection.delete_many({})

collection.insert_many(records)

print("Inserted records:", collection.count_documents({}))

Inserted records: 8000


In [11]:
# ============================================================
# 11. TEST BASIC QUERIES
# ============================================================

print("Total records:", collection.count_documents({}))

print("Training records:", collection.count_documents({"split": "training"}))
print("Validation records:", collection.count_documents({"split": "validation"}))
print("Test records:", collection.count_documents({"split": "test"}))

Total records: 8000
Training records: 6400
Validation records: 800
Test records: 800


In [12]:
# ============================================================
# 12. COUNT TRACKS BY GENRE
# ============================================================

genre_counts = collection.aggregate([
    {
        "$group": {
            "_id": "$genre.genre_top",
            "count": {"$sum": 1}
        }
    },
    {
        "$sort": {"count": -1}
    }
])

for item in genre_counts:
    print(item)

{'_id': 'Instrumental', 'count': 1000}
{'_id': 'Experimental', 'count': 1000}
{'_id': 'Pop', 'count': 1000}
{'_id': 'Hip-Hop', 'count': 1000}
{'_id': 'Folk', 'count': 1000}
{'_id': 'Electronic', 'count': 1000}
{'_id': 'Rock', 'count': 1000}
{'_id': 'International', 'count': 1000}


In [13]:
# ============================================================
# 13. VIEW ONE SAMPLE RECORD
# ============================================================

sample = collection.find_one()

sample

{'_id': ObjectId('69f2f1163e09dd500b484235'),
 'track_id': 2,
 'subset': 'small',
 'split': 'training',
 'genre': {'genre_top': 'Hip-Hop', 'genres': '[21]', 'genres_all': '[21]'},
 'track_metadata': {'title': 'Food',
  'duration': 168.0,
  'listens': 1293,
  'interest': 4656,
  'language_code': 'en',
  'license': 'Attribution-NonCommercial-ShareAlike 3.0 International'},
 'artist_metadata': {'artist_id': 1,
  'artist_name': 'AWOL',
  'artist_location': 'New Jersey'},
 'album_metadata': {'album_id': 1,
  'album_title': 'AWOL - A Way Of Life',
  'album_type': 'Album'},
 'audio_path': '../data/raw/audio/fma_small\\000\\000002.mp3',
 'features': {'chroma_cens_kurtosis_01': 7.1806526184,
  'chroma_cens_kurtosis_02': 5.2303090096,
  'chroma_cens_kurtosis_03': 0.24932080507,
  'chroma_cens_kurtosis_04': 1.3476201296,
  'chroma_cens_kurtosis_05': 1.4824777842,
  'chroma_cens_kurtosis_06': 0.53137123585,
  'chroma_cens_kurtosis_07': 1.4815930128,
  'chroma_cens_kurtosis_08': 2.691454649,
  'chr